# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset elements including record sets, fields, and columns are referenced by their `@id`, ensuring precise and unambiguous access.

### Dataset Source
The dataset is described by its Croissant schema, available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install `mlcroissant` if it's not already available
!pip install -U mlcroissant

## 1. Data Loading

Let's load the metadata and inspect the dataset with `mlcroissant`.

We will refer to dataset elements through their `@id` (Croissant standard) throughout this notebook.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata and records
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an object, not a dict

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets and their fields. We'll enumerate record set and field `@id`s, which you must use for accurate data access in the `mlcroissant` API.

- **Record set `@id`** = The unique identifier of a logical table of records in the dataset.
- **Field `@id`** = A unique identifier for each column/field in a record set.

In [ ]:
# List all available record sets by @id and name
record_sets = list(dataset.record_sets())
print("Available record sets:")
for rs in record_sets:
    print(f"  Record Set @id: {rs.id}  |  Name: {rs.name}")

# For this dataset, get fields and columns for the main record set
if record_sets:
    # We'll use the first record set (likely the main table)
    main_record_set = record_sets[0]
    print(f"\nFields in record set '{main_record_set.name}' (@id: {main_record_set.id}):")
    for f in main_record_set.fields:
        print(f"  Field @id: {f.id}  |  Name: {f.name}  |  DataType: {getattr(f, 'data_type', None)}")

    # List columns if they are specified
    if hasattr(main_record_set, 'columns') and main_record_set.columns:
        print("\nColumns (for file mapping, if applicable):")
        for col in main_record_set.columns:
            print(f"  Column @id: {col.id}  |  Name: {col.name}")

## 3. Data Extraction

Load data from the main record set into a pandas DataFrame for analysis using its `@id`.

In [ ]:
# Use main record set by its @id as found in the overview
# If your dataset has more record sets, you can extend 'record_set_ids' here
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # All records as list of dicts keyed by field @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show columns of the first/main record set
main_record_set_id = record_set_ids[0]
print(f"Fields (columns) for record set '{main_record_set_id}':")
print(dataframes[main_record_set_id].columns.tolist())

# Preview the first few records
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps (e.g., filtering on a numeric field such as patient age or diagnosis interval, normalization, grouping). **All field access must use the field's `@id`.**

_You can find the field `@id` from the output of section 2. Typical numeric @id might look like `http://mlcommons.org/croissant/field/Age` or similar._

In [ ]:
# Choose a realistic numeric field @id from the dataset
# Hypothetical examples for illustration - replace with actual @id from above if different.
from numpy import number

main_df = dataframes[main_record_set_id]

# Display all column @id and pick a numeric one
print("Available field @id's:")
print(main_df.columns.tolist())

# Try to pick a field that is numeric, fallback to any if not found
import re
# Candidates (replace with your dataset's actual @id, e.g. if Age is @id 'http://mlcommons.org/croissant/field/Age')
candidate_numeric_ids = [col for col in main_df.columns if re.search(r'(Age|Interval|Years|Duration|Number|Count|Distance|Level|Stage|Tumor)', col, re.I)]

if candidate_numeric_ids:
    numeric_field_id = candidate_numeric_ids[0]
else:
    # fallback, just use the first column
    numeric_field_id = main_df.columns[0]

print(f"Using numeric field @id: {numeric_field_id}")

# Ensure the field is numeric, coerce if necessary
main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
# Filter: keep only records with value > threshold
threshold = main_df[numeric_field_id].dropna().median()  # Use median as threshold example
filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
print(f"\nFiltered records with {numeric_field_id} > {threshold} (median):")
print(filtered_df.head())

# Normalize
field_norm = f"{numeric_field_id}_normalized"
filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}' for filtered records (z-score):")
print(filtered_df[[numeric_field_id, field_norm]].head())

# Try grouping by a categorical field (e.g., Sex, MSI status, Tumor Location)
candidate_group_fields = [col for col in main_df.columns if re.search(r'(Sex|Gender|MSI|Location|Group|Type|Status)', col, re.I)]
if candidate_group_fields:
    group_field_id = candidate_group_fields[0]
    print(f"\nGrouping by field @id: {group_field_id}")
    if group_field_id in filtered_df:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean').reset_index()
        print("Grouped mean by categorical field:")
        print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization

Visualize distributions of the selected numeric field and, if available, how it varies by a key grouping (e.g., MSI status, anatomical region).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(main_df[numeric_field_id].dropna(), bins=15, kde=True)
plt.title(f"Distribution of field: {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If grouping field exists, plot boxplot
if candidate_group_fields and (candidate_group_fields[0] in main_df.columns):
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=main_df[candidate_group_fields[0]], y=main_df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {candidate_group_fields[0]}")
    plt.xlabel(candidate_group_fields[0])
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

We have successfully loaded and explored the FAIR² colorectal cancer survivors dataset using `mlcroissant`. By leveraging entity `@id`s, we provided robust, schema-driven access to all data and metadata fields. You can now further adapt this notebook for domain-specific analyses, modeling, or FAIR data workflows.

For more on the Croissant format and tools, see [mlcroissant documentation](https://github.com/mlcommons/croissant).